# Serve Recommendation System

This notebook uses the trained model to recommend serve choices under specific match contexts.

The recommendation score combines predicted win probability, historical serve performance, and sample-size reliability.

In [ ]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("../data/processed/table_tennis_serves_features.csv")
model = joblib.load("../models/serve_win_probability_model.pkl")
model_features = joblib.load("../models/model_features.pkl")

In [ ]:
current_context = {"game_number":2,"server_score":8,"receiver_score":7,"game_state":"neutral","opponent_skill_level":"advanced","opponent_style":"looper","side":"backhand_side"}
current_context

This context represents the information known before the serve. The recommendation system will evaluate possible serve options under this situation.

In [ ]:
serve_options = df[["serve_type","spin_type","spin_intensity","serve_length","placement_zone","toss_height","contact_point","intended_setup"]].drop_duplicates()
recommendation_df = serve_options.copy()
for key, value in current_context.items():
    recommendation_df[key] = value

In [ ]:
# Recreate engineered features
recommendation_df["score_margin"] = recommendation_df["server_score"] - recommendation_df["receiver_score"]
recommendation_df["total_points_played_in_game"] = recommendation_df["server_score"] + recommendation_df["receiver_score"]
recommendation_df["is_tied"] = (recommendation_df["server_score"] == recommendation_df["receiver_score"]).astype(int)
recommendation_df["is_trailing"] = (recommendation_df["server_score"] < recommendation_df["receiver_score"]).astype(int)
recommendation_df["is_leading"] = (recommendation_df["server_score"] > recommendation_df["receiver_score"]).astype(int)
recommendation_df["is_late_game"] = (recommendation_df["total_points_played_in_game"] >= 16).astype(int)
recommendation_df["is_deuce_or_later"] = ((recommendation_df["server_score"] >= 10) & (recommendation_df["receiver_score"] >= 10)).astype(int)
recommendation_df["is_game_point_for_server"] = ((recommendation_df["server_score"] >= 10) & (recommendation_df["server_score"] > recommendation_df["receiver_score"])).astype(int)
recommendation_df["is_game_point_against_server"] = ((recommendation_df["receiver_score"] >= 10) & (recommendation_df["receiver_score"] > recommendation_df["server_score"])).astype(int)
recommendation_df["serve_spin_combo"] = recommendation_df["serve_type"] + "_" + recommendation_df["spin_type"]
recommendation_df["serve_length_spin_combo"] = recommendation_df["serve_length"] + "_" + recommendation_df["spin_type"]
recommendation_df["serve_placement_combo"] = recommendation_df["serve_type"] + "_" + recommendation_df["placement_zone"]
recommendation_df["full_serve_combo"] = recommendation_df["serve_type"] + "_" + recommendation_df["spin_type"] + "_" + recommendation_df["serve_length"] + "_" + recommendation_df["placement_zone"]
recommendation_df["is_heavy_spin"] = (recommendation_df["spin_intensity"] >= 3).astype(int)
recommendation_df["is_low_spin"] = (recommendation_df["spin_intensity"] <= 1).astype(int)

In [ ]:
combo_summary = df.groupby("full_serve_combo").agg(combo_attempts=("point_won", "count"), combo_win_rate=("point_won", "mean")).reset_index()
recommendation_df = recommendation_df.merge(combo_summary, on="full_serve_combo", how="left")
recommendation_df["combo_attempts"] = recommendation_df["combo_attempts"].fillna(0)
recommendation_df["combo_win_rate"] = recommendation_df["combo_win_rate"].fillna(df["point_won"].mean())
recommendation_df["combo_reliability"] = np.minimum(recommendation_df["combo_attempts"] / 30, 1)

In [ ]:
recommendation_df["predicted_win_probability"] = model.predict_proba(recommendation_df[model_features])[:, 1]

In [ ]:
recommendation_df["recommendation_score"] = 0.70 * recommendation_df["predicted_win_probability"] + 0.20 * recommendation_df["combo_win_rate"] + 0.10 * recommendation_df["combo_reliability"]

The recommendation score combines three pieces of information:

1. Model-predicted win probability
2. Historical win rate for that serve combination
3. Reliability based on sample size

This prevents the system from over-recommending serve combinations that performed well only once or twice.

In [ ]:
top_recommendations = recommendation_df.sort_values("recommendation_score", ascending=False)[["serve_type","spin_type","spin_intensity","serve_length","placement_zone","intended_setup","predicted_win_probability","combo_win_rate","combo_attempts","combo_reliability","recommendation_score"]].head(10)
top_recommendations

In [ ]:
def explain_recommendation(row):
    reasons = []
    if row["predicted_win_probability"] >= 0.60: reasons.append("high predicted win probability")
    if row["combo_reliability"] >= 0.70: reasons.append("reliable historical sample")
    if row["combo_win_rate"] >= 0.60: reasons.append("strong historical win rate")
    if row["spin_intensity"] >= 3: reasons.append("uses heavy spin")
    return ", ".join(reasons)

top_recommendations["reason"] = top_recommendations.apply(explain_recommendation, axis=1)
top_recommendations

In [ ]:
contexts = [{"label":"Neutral point vs looper","game_number":1,"server_score":4,"receiver_score":4,"game_state":"neutral","opponent_skill_level":"advanced","opponent_style":"looper","side":"backhand_side"},{"label":"Late-game pressure vs attacker","game_number":3,"server_score":9,"receiver_score":9,"game_state":"pressure","opponent_skill_level":"advanced","opponent_style":"attacker","side":"forehand_side"},{"label":"Trailing vs chopper","game_number":2,"server_score":6,"receiver_score":9,"game_state":"trailing","opponent_skill_level":"advanced","opponent_style":"chopper","side":"backhand_side"}]
contexts

## Summary

This notebook converts the predictive model into a serve recommendation system. For each match context, the system ranks possible serves using predicted win probability, historical performance, and sample-size reliability.

The recommendation system should be treated as a decision-support tool rather than an automatic answer. As more match data is collected, the reliability of the recommendations should improve.